# 25 — Production NLP: Contracts, Monitoring, Drift & Retraining

**Learning objective.** Define production inference contracts and monitor feature/label/quality drift with simple measurable signals.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**live requests + delayed labels → health/drift/quality signals → investigate → operate/retrain**

Follow the information transformation first; treat the API as an implementation detail.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Input vocabulary shifts | OOV/embedding distribution changes | model confidence/quality may degrade |
| Class prior shifts | prediction distribution changes | capacity/routing load changes even if conditional accuracy is stable |
| Threshold changes | automation coverage changes | human-review load and accepted-case quality trade off |

> Write down what should move downstream before changing a control.

## Think before running the next cell

1. If OOV rate rises but labeled F1 stays stable, should you retrain immediately?
2. Can prediction-distribution drift occur without concept drift?

### When to use
Use monitoring from day one for systems whose inputs, users or upstream products can change.

### When not to use / caution
Do not retrain solely because a drift statistic crossed a threshold—investigate quality and business context.

### Debugging lens
Separate operational failures, data drift, label/prior shift and concept drift; each has a different response.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
train_lengths=np.array([5,7,8,6,10,9,7,6,8,5,9,7])
prod_lengths=np.array([8,12,15,9,14,13,12,10,11,16,9,13])
print('train mean tokens:',round(train_lengths.mean(),2))
print('prod mean tokens :',round(prod_lengths.mean(),2))
print('relative shift   :',round(prod_lengths.mean()/train_lengths.mean()-1,3))

train mean tokens: 7.25
prod mean tokens : 11.83
relative shift   : 0.632


In [3]:
def psi(expected, actual, bins=5):
    cuts=np.quantile(expected,np.linspace(0,1,bins+1))
    cuts[0]-=1e-9; cuts[-1]+=1e-9
    e=np.histogram(expected,bins=cuts)[0]/len(expected)
    a=np.histogram(actual,bins=cuts)[0]/len(actual)
    e=np.clip(e,1e-6,None); a=np.clip(a,1e-6,None)
    return float(np.sum((a-e)*np.log(a/e)))
print('PSI(length):',round(psi(train_lengths,prod_lengths),3))

PSI(length): 7.173


## Production checklist
**Input:** schema, encoding, language, max length, PII handling.
**Model:** immutable artifact/version, tokenizer/vectorizer version, deterministic fallback.
**Serving:** latency percentiles, throughput, batching, timeouts, resource saturation.
**Quality:** prediction distribution, confidence/calibration, human overrides, slice metrics when labels arrive.
**Drift:** vocabulary/OOV, length, language, embedding/feature distribution, label/prior shift.
**Governance:** audit logs, retention, model card, data lineage, rollback, approval gates.
**Retraining:** explicit trigger, comparable holdout, regression tests, canary/shadow evaluation.

In [4]:
contract={
 'input':{'text':'non-empty UTF-8 string','max_chars':10000},
 'output':{'label':'controlled vocabulary','model_version':'required'},
 'monitor':['latency_p95','error_rate','input_length','label_distribution','human_override_rate']}
print(json.dumps(contract,indent=2))

{
  "input": {
    "text": "non-empty UTF-8 string",
    "max_chars": 10000
  },
  "output": {
    "label": "controlled vocabulary",
    "model_version": "required"
  },
  "monitor": [
    "latency_p95",
    "error_rate",
    "input_length",
    "label_distribution",
    "human_override_rate"
  ]
}


In [5]:

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(train_lengths,bins=6,alpha=.6,label='training')
ax.hist(prod_lengths,bins=6,alpha=.6,label='production')
ax.set_xlabel('token length'); ax.set_ylabel('count'); ax.set_title('Input-length distribution shift')
ax.legend(); plt.tight_layout(); plt.show()


[static visualization generated successfully during execution; rerun in Jupyter/VS Code to display]


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Define a production inference contract
- Separate data drift, concept drift and operational health